# EXPLORATION - NOT MAIN PORTFOLIO
# DATE: 4/3/26

# FINDINGS TO BE ADDED TO MAIN:
- Baseline random forest has the best performance
- Applying SMOTE before the tuning ruined the performance scores
- Future work should incorporate SMOTE or other sampling technique inside the CV instead of before.

## Goal
Testing several parameters for the model selected on the previous experiment to find the best parameter for the model. Final outcome is to decide which parameter to use for the final model

## Setup

In [1]:
import pandas as pd

data = pd.read_csv('creditcard.csv')

data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


as decided in previous experiments, we will remove some of the v features, use SMOTE resampling, and we use random forest for the model.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, \
confusion_matrix

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from datetime import datetime

In [3]:
#remove v
low_score_v = ['V22', 'V23', 'V25', 'V26', 'V28', 'V15', 'V24', 'V13', 'V27']

reduced_v = data.drop(columns=low_score_v)
X = reduced_v.drop('Class', axis=1)
y = reduced_v['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

since hyperparameter tuning will only be using training data, the SMOTE resampled will be added as a pipeline alongside the model, instead of doing it now

In [4]:
X_train.shape

(227845, 21)

## Tuning

### Baseline

the baseline for comparison will use the simplest parameter as used in exp 2 & 3. Model with tuned parameter will be compared to this baseline

In [5]:
base_model = RandomForestClassifier(n_estimators=100, random_state=1, n_jobs=-1)
smote = SMOTE(random_state=1)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train,y_train)

start_time = time.time()
base_model.fit(X_train_resampled, y_train_resampled)
y_pred_base = base_model.predict(X_test)
baseline_time = time.time() - start_time

print('Baseline Performance')
print(classification_report(y_test, y_pred_base))
print('Confusion Matrix')
print(confusion_matrix(y_test,y_pred_base))

baseline_metrics = {
    'precision': precision_score(y_test, y_pred_base),
    'recall': recall_score(y_test, y_pred_base),
    'f1': f1_score(y_test, y_pred_base),
    'train_time': baseline_time
}

Baseline Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.89      0.88        98

    accuracy                           1.00     56962
   macro avg       0.94      0.94      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56852    12]
 [   11    87]]


In [6]:
print(f"\nBaseline Performance:")
print(f"  Precision:     {baseline_metrics['precision']:.4f}")
print(f"  Recall:        {baseline_metrics['recall']:.4f}")
print(f"  F1-Score:      {baseline_metrics['f1']:.4f}")
print(f"  Training Time: {baseline_metrics['train_time']:.2f}s")


Baseline Performance:
  Precision:     0.8788
  Recall:        0.8878
  F1-Score:      0.8832
  Training Time: 57.44s


baseline performance is already good enough but we will see if hyperparameter tuning can improve the performance even more. Even an improvement as slightly as 1% is already good enough to be consider

### Hyperparameter Tuning (NOT USED)
#### Parameters to Tune:
1. **n_estimators**: 100, 200, 300, 500
2. **max_depth**: 15, 20, 25, 30, None
3. **min_samples_split**: 2, 5, 10, 15
4. **max_features**: 'sqrt', 'log2', None
5. **bootstrap**: True, False

We will use GridSearchCV to do the tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True]
}

In [ ]:
rf_grid = GridSearchCV(
    estimator = RandomForestClassifier(random_state=1),
    param_grid = param_grid,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

# Train (this takes a while!)
print(f"\nStarting grid search at {datetime.now().strftime('%H:%M:%S')}...")

start_time = time.time()
rf_grid.fit(X_train, y_train)
grid_search_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Grid search complete! Time taken: {grid_search_time/60:.1f} minutes")

In [ ]:
print('Best Parameter:')
for param, value in rf_grid.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest Cross-Validation Recall Score: {rf_grid.best_score_:.4f}")
print(f"Baseline Recall Score:               {baseline_metrics['recall']:.4f}")
print(f"Improvement:                     {rf_grid.best_score_ - baseline_metrics['recall']:+.4f}")

### Evaluate on Test Set

In [ ]:
rf_grid.best_params_

In [ ]:
best_rf = rf_grid.best_estimator_

y_pred_tuned = best_rf.predict(X_test_scaled)

print('Tuned Performance')
print(classification_report(y_test, y_pred_tuned))
print('Tuned Confusion Matrix')
print(confusion_matrix(y_test,y_pred_tuned))

tuned_metrics = {
    'precision': precision_score(y_test, y_pred_tuned),
    'recall': recall_score(y_test, y_pred_tuned),
    'f1': f1_score(y_test, y_pred_tuned)
}

In [ ]:
print("Tuned Model Performance:")
print(f"  Precision:  {tuned_metrics['precision']:.4f}")
print(f"  Recall:     {tuned_metrics['recall']:.4f}")
print(f"  F1-Score:   {tuned_metrics['f1']:.4f}")

I've tried grid search several times and I realized my current machine is not really the best for hyperparameter tuning as it takes too long to even finish one. I've tried from 9000 fits to around 1000 fits and still no luck. From here on, I'll just do random search to find the best params and probably do grid search with value near the best parameter for validation.

### Hyperparameter Tuning #2

#### Parameters to Tune:
1. **n_estimators**: 100, 200, 300
2. **max_depth**: 10, 15, 20
3. **min_samples_split**: 2, 5, 10
4. **max_features**: 'sqrt', 'log2'
5. **bootstrap**: True

We will use RandomizedSearchCV to do the tuning

In [8]:
pipeline = Pipeline([
    ("smote", SMOTE(random_state=1)),
    ("model", RandomForestClassifier(random_state=1))
])

we add SMOTE in the model pipeline

In [9]:
param_random = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 15, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__max_features': ['sqrt', 'log2'],
    'model__bootstrap': [True]
}

since we use a pipeline, and not a single model, we need to specify which pipeline the parameters belong to, by putting model__

In [10]:
rf_random = RandomizedSearchCV(
    estimator = pipeline,
    param_distributions = param_random,
    n_iter=30,
    cv=3,
    scoring='recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    random_state=1
)

# Train (this takes a while!)
print(f"\nStarting random search at {datetime.now().strftime('%H:%M:%S')}...")

start_time = time.time()
rf_random.fit(X_train, y_train)
random_search_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Random search complete! Time taken: {random_search_time/60:.1f} minutes")


Starting random search at 14:00:41...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Random search complete! Time taken: 98.5 minutes


In [11]:
print('Best Parameter:')
for param, value in rf_random.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest Cross-Validation Recall Score: {rf_random.best_score_:.4f}")
print(f"Baseline Recall Score:               {baseline_metrics['recall']:.4f}")
print(f"Improvement:                     {rf_random.best_score_ - baseline_metrics['recall']:+.4f}")

Best Parameter:
  model__n_estimators : 300
  model__min_samples_split: 10
  model__max_features : log2
  model__max_depth    : 10
  model__bootstrap    : True

Best Cross-Validation Recall Score: 0.8375
Baseline Recall Score:               0.8878
Improvement:                     -0.0502


The best parameter after tuning with RandomSearch actually has a lower recall score than the baseline, with a decrease of 5%. We will try to validate this result first with gridsearch with parameters near the best.

#### Validate with GridSearch

In [12]:
print(rf_random.best_params_)

{'model__n_estimators': 300, 'model__min_samples_split': 10, 'model__max_features': 'log2', 'model__max_depth': 10, 'model__bootstrap': True}


In [14]:
pipeline = Pipeline([
    ("smote", SMOTE(random_state=1)),
    ("model", RandomForestClassifier(random_state=1))
])

In [15]:
param_grid = {
    'model__n_estimators': [250, 300, 350],
    'model__max_depth': [7, 10, 13],
    'model__min_samples_split': [8, 10, 12],
    'model__max_features': ['log2'],
    'model__bootstrap': [True]
}

In [16]:
rf_grid = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    cv=3,
    scoring= 'recall',
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

# Train
print(f"\nStarting grid search at {datetime.now().strftime('%H:%M:%S')}...")

start_time = time.time()
rf_grid.fit(X_train, y_train)
grid_search_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"Grid search complete! Time taken: {grid_search_time/60:.1f} minutes")


Starting grid search at 17:40:19...
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Grid search complete! Time taken: 98.5 minutes


In [17]:
#let's see if the best params from random search is on the top
results = pd.DataFrame(rf_grid.cv_results_)

results.sort_values("mean_test_score", ascending=False).head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__bootstrap,param_model__max_depth,param_model__max_features,param_model__min_samples_split,param_model__n_estimators,params,...,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score
1,785.200363,5.261822,2.533396,0.029450,True,7,log2,8,300,"{'model__bootstrap': True, 'model__max_depth':...",...,0.839695,0.870229,0.865429,0.019352,1,0.908397,0.942966,0.904943,0.918769,0.017168
2,919.907400,19.419823,2.968300,0.072642,True,7,log2,8,350,"{'model__bootstrap': True, 'model__max_depth':...",...,0.839695,0.870229,0.865429,0.019352,1,0.908397,0.942966,0.904943,0.918769,0.017168
6,654.987404,1.265838,2.436041,0.426861,True,7,log2,12,250,"{'model__bootstrap': True, 'model__max_depth':...",...,0.839695,0.870229,0.865429,0.019352,1,0.908397,0.946768,0.908745,0.921303,0.018007
0,654.089219,6.494866,2.141149,0.062709,True,7,log2,8,250,"{'model__bootstrap': True, 'model__max_depth':...",...,0.839695,0.862595,0.862885,0.019054,4,0.908397,0.946768,0.901141,0.918769,0.020019
3,663.508668,13.921439,2.232875,0.035519,True,7,log2,10,250,"{'model__bootstrap': True, 'model__max_depth':...",...,0.839695,0.862595,0.862885,0.019054,4,0.908397,0.942966,0.908745,0.920036,0.016214


In [18]:
rf_grid.best_params_

{'model__bootstrap': True,
 'model__max_depth': 7,
 'model__max_features': 'log2',
 'model__min_samples_split': 8,
 'model__n_estimators': 300}

In [19]:
rf_grid.best_score_

0.8654291001619246

from all these parameters, at least we can conclude that the model is quite stable near the best parameter. However, all those best score of 87% recall score is still lower than the baseline. Next is to evaluate whether the model is actually reliable, by using the test set.

#### Evaluate on Test Set

In [21]:
#best parameter from the grid search
best_rf = RandomForestClassifier(n_estimators=300, min_samples_split = 8, max_features = 'log2',
                                max_depth=7, bootstrap=True, random_state=1)

#use resampled set
best_rf.fit(X_train_resampled, y_train_resampled)
y_pred_tuned = best_rf.predict(X_test)

print('Tuned Performance')
print(classification_report(y_test, y_pred_tuned))
print('Tuned Confusion Matrix')
print(confusion_matrix(y_test,y_pred_tuned))

tuned_metrics = {
    'precision': precision_score(y_test, y_pred_tuned),
    'recall': recall_score(y_test, y_pred_tuned),
    'f1': f1_score(y_test, y_pred_tuned)
}

Tuned Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.26      0.91      0.41        98

    accuracy                           1.00     56962
   macro avg       0.63      0.95      0.70     56962
weighted avg       1.00      1.00      1.00     56962

Tuned Confusion Matrix
[[56616   248]
 [    9    89]]


In [22]:
#try the next best parameter from the grid search
test_rf_1 = RandomForestClassifier(n_estimators=250, min_samples_split = 12, max_features = 'log2',
                                max_depth=7, bootstrap=True, random_state=1)

#use resampled set
test_rf_1.fit(X_train_resampled, y_train_resampled)
y_pred_tuned_2 = best_rf.predict(X_test)

print('Tuned Performance')
print(classification_report(y_test, y_pred_tuned_2))
print('Tuned Confusion Matrix')
print(confusion_matrix(y_test,y_pred_tuned_2))

tuned_metrics_2 = {
    'precision': precision_score(y_test, y_pred_tuned_2),
    'recall': recall_score(y_test, y_pred_tuned_2),
    'f1': f1_score(y_test, y_pred_tuned_2)
}

Tuned Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.26      0.91      0.41        98

    accuracy                           1.00     56962
   macro avg       0.63      0.95      0.70     56962
weighted avg       1.00      1.00      1.00     56962

Tuned Confusion Matrix
[[56616   248]
 [    9    89]]


we can see that the tuned model actually performed worse than the baseline, even though the recall score is improved by 3%, the precision and f1 score are too little from having too many false positives.

In [24]:
# Create comparison table
results_df = pd.DataFrame({
    'Experiment': ['Baseline', 'Best Parameter', '2nd Best'],
    'Precision': [baseline_metrics['precision'], tuned_metrics['precision'], 
                  tuned_metrics_2['precision']],
    'Recall': [baseline_metrics['recall'], tuned_metrics['recall'], 
               tuned_metrics_2['recall']],
    'F1': [baseline_metrics['f1'], tuned_metrics['f1'], 
           tuned_metrics_2['f1']]
})

print(results_df)

       Experiment  Precision    Recall        F1
0        Baseline   0.878788  0.887755  0.883249
1  Best Parameter   0.264095  0.908163  0.409195
2        2nd Best   0.264095  0.908163  0.409195


based on these calculations, it is concluded that the best performance are the baseline model, Random Forest with the default parameters. Model performance using the tuned parameters resulted in both lower f1 score and precision, with very low scores.

## Summary

### Final Summary

#### Optimization Results
**Random Search Configuration:**
- Parameter combination: 54
- Cross-validation folds: 3
- Optimiazation metrics: Recall score
- Total training time: 108.2 minutes

**Best Parameters Found:**
{'n_estimators': 300, 'min_samples_split': 8, 'max_features': 'log2', 'max_depth': 7, 'bootstrap': True}

#### Key Findings
The tuned model has a worse performance score than the baseline even with a higher recall (recall +3%, f1-score -47%, precision -61%) further proved that the baseline were already optimal for this dataset. This also validates that Random Forest model with reasonable parameters is highly effective for this fraud detection tasks, even with little to no adjustments.

#### Fraud Detection
- **Frauds Caught**: 87 out of 98 (88.8%)
- **Frauds Missed**: 11 (11.2%)
- **False Alarm**: 12 out of 56852

### Notes for next Project
1. Try tuning the threshold as well
Random forest is not a binary model but a probability one. It outputs a probability from 0 to 1 and the threshold convert it to 1 or 0. Above threshold it is 1 and below threshold it is 0. Changing the threshold might be beneficial since with higher threshold we might get higher precision.

2. Use f1-score for the score method instead of recall
Since we only use recall score for the tuning, the model might become to "aggressive" and try to catch as many positives as possible. And it is proven that the recall actually improved significantly (3%). However, it does produce a lot of false positives in the process, so this method might not be the most beneficial. Using f1-score or precision may produce a better model with better performance

### Notes to Self

after multiple attempt across several days, turns out the problem is pretty trivial. I was using too high parameter value, n_estimators = 300, 500 and max_depths = 25, 30, None, which slows things down by a lot. After removing those high value my code run fine with expectable run time. Never again I will use those values. Turns out my laptop is fine, its the parameter that is too much